### Inspect scraped data_files

In [1]:
import pandas as pd
from itertools import combinations

In [2]:
logboek_df = pd.read_csv('logboek.csv', sep=';')
countries = logboek_df["land"].tolist()
leagues = logboek_df["competitienaam"].tolist()
seasons = logboek_df["jaar"].tolist()

In [3]:
print(countries)

['england', 'mexico', 'japan', 'iran', 'south-korea', 'australia', 'morocco', 'senegal', 'nigeria', 'algeria', 'egypt', 'ivory-coast', 'usa', 'canada', 'panama', 'argentina', 'brazil', 'colombia', 'uruguay', 'ecuador', 'paraguay', 'austria', 'belgium', 'cyprus', 'czech-republic', 'france', 'germany', 'hungary', 'ireland', 'italy', 'luxembourg', 'malta', 'netherlands', 'northern-ireland', 'norway', 'poland', 'portugal', 'romania', 'russia', 'scotland', 'slovakia', 'spain', 'switzerland', 'turkey', 'wales', 'croatia', 'denmark', 'ukraine', 'serbia', 'sweden']


### checked up untill index 0

In [4]:
def filter_rounds(df):
    return_df = df.copy()
    return_df = return_df[return_df["round"].str.startswith("Round ")].reset_index(drop=True)
    return return_df

In [5]:
def filter_playoffs(df, start_round_playoffs):
    return_df = df.copy()
    return_df = return_df[return_df["round"].str[6:].astype(int) < start_round_playoffs].reset_index(drop=True)
    return return_df

In [6]:
def filter_on_date(df, start_date, keep_before = True):
    """ 
    Format start_date: "YYYY-MM-DD"
    """
    return_df = df.copy()
    start_date = pd.to_datetime(start_date)
    
    return_df["date"] = pd.to_datetime(return_df["date"], errors='coerce')
    if keep_before:
        return_df = return_df[(return_df["date"] < start_date)].reset_index(drop=True)
    else:
        return_df = return_df[(return_df["date"] >= start_date)].reset_index(drop=True)
    return return_df

In [30]:
def fill_in_forfaits(df):
    return_df = df.copy()
    # Identify matches with missing scores
    try:
        index_forfait = return_df[return_df["team_home"] == "Awrd"].index[0]
        parts = return_df.loc[index_forfait, "full_text"].split(" | ")
        return_df.loc[index_forfait, "team_home"] = parts[2]
        return_df.loc[index_forfait, "team_away"] = parts[3]
        return return_df
    except:
        return df


In [8]:
def remove_round_prefix(df):
    return_df = df.copy()
    return_df["round"] = return_df["round"].apply(lambda x: x[6:])
    return return_df

In [9]:
def check_specific_match(df, team1, team2):
    return df[((df["team_home"] == team1) & (df["team_away"] == team2)) | ((df["team_home"] == team2) & (df["team_away"] == team1))]

In [10]:
def check_rounds(df):
    rounds = df["round"].unique().tolist()
    counted_rounds = df["round"].value_counts().to_dict()

    # check if all rounds appear the same amount of times
    rounds_appear_same_amount = len(set(counted_rounds.values())) == 1

    # check if rounds start with "Round"
    rounds_start_with_round = all(r.startswith("Round") for r in rounds)

    print(f"Rounds: {rounds}")
    print(f"Only normal rounds: {rounds_start_with_round}")
    print(f"All rounds appear same amount of times: {rounds_appear_same_amount}")

In [11]:
def check_teams(df):
    home_teams = df["team_home"].unique().tolist()
    away_teams = df["team_away"].unique().tolist()

    all_teams = set(home_teams) | set(away_teams)
    print(f"Teams: {all_teams}")
    print(f"Number of teams: {len(all_teams)}")

    # check if all teams appear in both home and away the same amount of times
    home_team_counts = df["team_home"].value_counts().to_dict()
    away_team_counts = df["team_away"].value_counts().to_dict()
    teams_appear_same_amount = all(home_team_counts.get(team, 0) == away_team_counts.get(team, 0) for team in all_teams)
    print(f"All teams appear same amount of times in home and away: {teams_appear_same_amount}")
    if teams_appear_same_amount:
        print(f"Each team appears {home_team_counts[home_teams[0]]} times in home and away")


In [12]:
def check_games(df):
    teams = sorted(set(df['team_home'].unique()) | set(df['team_away'].unique()))
    
    # Build symmetric n x n matrix
    matrix = pd.DataFrame(0, index=teams, columns=teams)
    
    for _, row in df.iterrows():
        h, a = row['team_home'], row['team_away']
        matrix.loc[h, a] += 1
        matrix.loc[a, h] += 1  # symmetric: home/away doesn't matter
    
    # Extract counts for all unique pairs (upper triangle)
    pair_counts = {
        (t1, t2): matrix.loc[t1, t2]
        for t1, t2 in combinations(teams, 2)
    }
    
    unique_counts = set(pair_counts.values())
    is_balanced = len(unique_counts) == 1 and 0 not in unique_counts
    
    print("=== Game Balance Check ===\n")
    print("Head-to-head matrix:")
    print(matrix.to_string())
    print()
    
    count_freq = {}
    for v in pair_counts.values():
        count_freq[v] = count_freq.get(v, 0) + 1
    
    if is_balanced:
        times = unique_counts.pop()
        print(f"Balanced: every pair plays exactly {times} time(s).")
    else:
        print(f"Not balanced. Distribution of matchup counts: {count_freq}")
        if 0 in unique_counts:
            missing = [f"{t1} vs {t2}" for (t1, t2), v in pair_counts.items() if v == 0]
            print(f"   Never played: {missing[:10]}{'...' if len(missing) > 10 else ''}")
        modal = max(set(pair_counts.values()), key=list(pair_counts.values()).count)
        odd = {f"{k[0]} vs {k[1]}": v for k, v in pair_counts.items() if v != modal}
        if odd:
            print(f"   Odd counts: {odd}")
    

In [114]:
# Checked untill index 10, egypt
index = 49
country = countries[index]
league = leagues[index]
season = seasons[index]
print(country)
print(league)
print(season)
filename = f"{country}_{league}_{season}.csv"
df = pd.read_csv(f"./scraped_data/{filename}")
df = fill_in_forfaits(df)
df

sweden
allsvenskan
2025


,round,date,team_home,team_away,full_text
0,Final,29.11.2025,Norrkoping,Orgryte,29.11.2025 | Norrkoping | Orgryte | Winner: Or...
1,Final,22.11.2025,Orgryte,Norrkoping,22.11.2025 | Orgryte | Norrkoping | 3 | 0
2,Round 30,09.11.2025,AIK,Halmstad,09.11.2025 | AIK | Halmstad | 0 | 2
3,Round 30,09.11.2025,Brommapojkarna,Degerfors,09.11.2025 | Brommapojkarna | Degerfors | 1 | 3
4,Round 30,09.11.2025,Goteborg,Norrkoping,09.11.2025 | Goteborg | Norrkoping | 2 | 0
...,...,...,...,...,...
237,Round 1,30.03.2025,Norrkoping,Oster,30.03.2025 | Norrkoping | Oster | 4 | 3
238,Round 1,30.03.2025,Elfsborg,Mjallby,30.03.2025 | Elfsborg | Mjallby | 2 | 2
239,Round 1,30.03.2025,Hammarby,Goteborg,30.03.2025 | Hammarby | Goteborg | 4 | 0
240,Round 1,29.03.2025,Hacken,Brommapojkarna,29.03.2025 | Hacken | Brommapojkarna | 2 | 0


In [94]:
# First do the checks, then do the filtering and removing of round prefix if necessary

check_rounds(df)
check_teams(df)
check_games(df)

Rounds: ['Round 36', 'Round 35', 'Round 34', 'Round 33', 'Round 32', 'Round 31', 'Round 30', 'Round 29', 'Round 28', 'Round 27', 'Round 26', 'Round 25', 'Round 24', 'Round 23', 'Round 22', 'Round 21', 'Round 20', 'Round 19', 'Round 18', 'Round 17', 'Round 16', 'Round 15', 'Round 14', 'Round 13', 'Round 12', 'Round 11', 'Round 10', 'Round 9', 'Round 8', 'Round 7', 'Round 6', 'Round 5', 'Round 4', 'Round 3', 'Round 2', 'Round 1']
Only normal rounds: True
All rounds appear same amount of times: True
Teams: {'Din. Zagreb', 'Lok. Zagreb', 'Hajduk Split', 'Rijeka', 'Istra 1961', 'Sibenik', 'Varazdin', 'Gorica', 'Osijek', 'Slaven Belupo'}
Number of teams: 10
All teams appear same amount of times in home and away: True
Each team appears 18 times in home and away
=== Game Balance Check ===

Head-to-head matrix:
               Din. Zagreb  Gorica  Hajduk Split  Istra 1961  Lok. Zagreb  Osijek  Rijeka  Sibenik  Slaven Belupo  Varazdin
Din. Zagreb              0       4             4           4  

In [95]:
# write away cleaned df and adapt logboek
df = remove_round_prefix(df)
df.to_csv(f"./cleaned_data/{filename}", index=False)

In [115]:
df_only_rounds = filter_rounds(df)

In [116]:
check_rounds(df_only_rounds)
check_teams(df_only_rounds)
check_games(df_only_rounds)

Rounds: ['Round 30', 'Round 29', 'Round 28', 'Round 27', 'Round 26', 'Round 25', 'Round 24', 'Round 23', 'Round 22', 'Round 21', 'Round 20', 'Round 19', 'Round 18', 'Round 17', 'Round 16', 'Round 15', 'Round 14', 'Round 13', 'Round 12', 'Round 6', 'Round 11', 'Round 10', 'Round 9', 'Round 8', 'Round 7', 'Round 5', 'Round 4', 'Round 3', 'Round 2', 'Round 1']
Only normal rounds: True
All rounds appear same amount of times: True
Teams: {'Elfsborg', 'Oster', 'Degerfors', 'Sirius', 'Goteborg', 'Hacken', 'Norrkoping', 'GAIS', 'Brommapojkarna', 'Varnamo', 'Halmstad', 'AIK', 'Malmo FF', 'Mjallby', 'Djurgarden', 'Hammarby'}
Number of teams: 16
All teams appear same amount of times in home and away: True
Each team appears 15 times in home and away
=== Game Balance Check ===

Head-to-head matrix:
                AIK  Brommapojkarna  Degerfors  Djurgarden  Elfsborg  GAIS  Goteborg  Hacken  Halmstad  Hammarby  Malmo FF  Mjallby  Norrkoping  Oster  Sirius  Varnamo
AIK               0               2

In [117]:
# write away cleaned df and adapt logboek
df_only_rounds = remove_round_prefix(df_only_rounds)
df_only_rounds.to_csv(f"./cleaned_data/{filename}", index=False)

In [111]:
df_only_rounds = filter_playoffs(df_only_rounds, 31)

In [86]:
df_only_rounds

,round,date,team_home,team_away,full_text
0,Round 33,21.04.2025,Basel,Yverdon,21.04.2025 | Basel | Yverdon | 5 | 0
1,Round 33,21.04.2025,Lausanne,Lugano,21.04.2025 | Lausanne | Lugano | 2 | 0
2,Round 33,21.04.2025,Servette,Luzern,21.04.2025 | Servette | Luzern | 2 | 1
3,Round 33,21.04.2025,St. Gallen,Sion,21.04.2025 | St. Gallen | Sion | 1 | 0
4,Round 33,21.04.2025,Young Boys,Zurich,21.04.2025 | Young Boys | Zurich | 2 | 1
...,...,...,...,...,...
193,Round 1,21.07.2024,Luzern,Servette,21.07.2024 | Luzern | Servette | 1 | 2
194,Round 1,21.07.2024,Young Boys,Sion,21.07.2024 | Young Boys | Sion | 1 | 2
195,Round 1,20.07.2024,Winterthur,St. Gallen,20.07.2024 | Winterthur | St. Gallen | 1 | 0
196,Round 1,20.07.2024,Lugano,Grasshoppers,20.07.2024 | Lugano | Grasshoppers | 2 | 1


In [62]:
df_only_rounds[df_only_rounds["round"] == "Round 1"]

,round,date,team_home,team_away,full_text


In [74]:
df_filtered = filter_on_date(df_only_rounds, "2025-03-07")

C:\Users\simcosyn\AppData\Local\Temp\ipykernel_23268\1965196199.py:8: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return_df["date"] = pd.to_datetime(return_df["date"], errors='coerce')


In [75]:
df_filtered

,round,date,team_home,team_away,full_text
0,Round 22,2025-03-01,Banska Bystrica,Zilina,01.03.2025 | Banska Bystrica | Zilina | 1 | 2
1,Round 22,2025-03-01,Dun. Streda,Ruzomberok,01.03.2025 | Dun. Streda | Ruzomberok | 3 | 0
2,Round 22,2025-03-01,Kosice,Michalovce,01.03.2025 | Kosice | Michalovce | 3 | 2
3,Round 22,2025-03-01,Podbrezova,Komarno,01.03.2025 | Podbrezova | Komarno | 2 | 2
4,Round 22,2025-03-01,Skalica,Trencin,01.03.2025 | Skalica | Trencin | 0 | 0
...,...,...,...,...,...
127,Round 1,2024-07-28,Ruzomberok,Banska Bystrica,28.07.2024 | Ruzomberok | Banska Bystrica | 1 | 1
128,Round 1,2024-07-28,Trnava,Trencin,28.07.2024 | Trnava | Trencin | 0 | 0
129,Round 1,2024-07-27,Komarno,Slovan Bratislava,27.07.2024 | Komarno | Slovan Bratislava | 1 | 4
130,Round 1,2024-07-27,Podbrezova,Zilina,27.07.2024 | Podbrezova | Zilina | 0 | 0


In [76]:
check_rounds(df_filtered)
check_teams(df_filtered)   
check_games(df_filtered)

Rounds: ['Round 22', 'Round 20', 'Round 21', 'Round 19', 'Round 18', 'Round 17', 'Round 12', 'Round 16', 'Round 15', 'Round 14', 'Round 13', 'Round 3', 'Round 5', 'Round 2', 'Round 11', 'Round 10', 'Round 9', 'Round 4', 'Round 8', 'Round 6', 'Round 7', 'Round 1']
Only normal rounds: True
All rounds appear same amount of times: True
Teams: {'Trencin', 'Komarno', 'Banska Bystrica', 'Podbrezova', 'Dun. Streda', 'Ruzomberok', 'Slovan Bratislava', 'Trnava', 'Skalica', 'Michalovce', 'Kosice', 'Zilina'}
Number of teams: 12
All teams appear same amount of times in home and away: True
Each team appears 11 times in home and away
=== Game Balance Check ===

Head-to-head matrix:
                   Banska Bystrica  Dun. Streda  Komarno  Kosice  Michalovce  Podbrezova  Ruzomberok  Skalica  Slovan Bratislava  Trencin  Trnava  Zilina
Banska Bystrica                  0            2        2       2           2           2           2        2                  2        2       2       2
Dun. Streda     

In [77]:
df_filtered = remove_round_prefix(df_filtered)
df_filtered.to_csv(f"./cleaned_data/{filename}", index=False)